In [ ]:
# Instalación de dependencias del entorno
!pip install uv
!uv pip install -r requirements.txt

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
# xarray es la librería principal para trabajar con datos multidimensionales (netCDF, Zarr)
import xarray as xr
# cKDTree permite búsquedas eficientes del punto más cercano en espacios n-dimensionales
from scipy.spatial import cKDTree
import pystac_client
import planetary_computer as pc
from datetime import date
from tqdm import tqdm
import os

In [ ]:
def load_terraclimate_dataset():
    """
    Carga el dataset TerraClimate desde Planetary Computer en formato Zarr.
    Zarr es un formato de almacenamiento en la nube optimizado para acceso parcial —
    solo se descargan los datos que se necesitan, no el dataset completo.
    """
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
    return ds


def filterg(ds, var):
    """
    Extrae una variable climática del dataset global y filtra solo los puntos
    dentro del área de Sudáfrica (bounding box) y el período 2011-2015.
    Devuelve una tabla plana con columnas Latitude, Longitude, Sample Date y el valor.
    """
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        # Convertir cada paso de tiempo a DataFrame y filtrar por coordenadas
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97)  & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")
    df_var_final['time'] = df_var_final['time'].astype(str)
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)
    return df_var_final


def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Para cada muestra del dataset de calidad del agua, asigna el valor climático
    del punto de la grilla TerraClimate más cercano en coordenadas y en tiempo.

    Usa cKDTree para la búsqueda espacial eficiente: convierte coordenadas a radianes
    para que la distancia euclidiana aproxime bien la distancia esférica.
    """
    sa_coords      = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    # Construir árbol de búsqueda y encontrar el punto más cercano
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date']      = pd.to_datetime(sa_df['Sample Date'],      dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        # Filtrar todos los registros del punto más cercano y elegir el mes más próximo
        subset = climate_df[
            (climate_df['Latitude']  == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]
        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    return pd.DataFrame({var_name: climate_values})

In [ ]:
# Cargar el dataset de calidad del agua con fechas y coordenadas de cada muestra
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
display(Water_Quality_df.head(5))
print(Water_Quality_df.shape)

In [ ]:
# Variables climáticas a extraer de TerraClimate:
# pet = evapotranspiración potencial (cuánta agua podría evaporarse si hubiera suficiente)
# ppt = precipitación mensual
# soil = humedad del suelo
# aet = evapotranspiración real (la que realmente ocurre, limitada por disponibilidad de agua)
VARIABLES = ['pet', 'ppt', 'soil', 'aet']

ds = load_terraclimate_dataset()
print("Variables disponibles en TerraClimate:", list(ds.data_vars))

result_dfs = {}

for var in VARIABLES:
    path = f"tc_{var}.csv"
    if os.path.exists(path):
        # Si ya se extrajo antes, reutilizar el archivo guardado
        print(f"Cargando {var.upper()} desde archivo")
        result_dfs[var] = pd.read_csv(path)
    else:
        print(f"Extrayendo {var.upper()}...")
        tc_grid = filterg(ds, var)
        var_df  = assign_nearest_climate(Water_Quality_df, tc_grid, var)
        var_df.to_csv(path, index=False)
        result_dfs[var] = var_df
        print(f"{var.upper()} guardado")

# Déficit hídrico = pet - aet: indica cuánta agua le falta al sistema
# Un déficit alto significa que se evapora más de lo que llueve — concentra minerales
result_dfs['water_deficit'] = pd.DataFrame({
    'water_deficit': result_dfs['pet']['pet'].values - result_dfs['aet']['aet'].values
})

# Verificar que todas las variables tienen el mismo número de filas
for var, df in result_dfs.items():
    assert len(df) == len(Water_Quality_df),         f"Desfase en {var}: {len(df)} vs {len(Water_Quality_df)}"
    print(f"{var}: {len(df)} filas OK")

In [ ]:
# Combinar todas las variables en una sola tabla alineada con el dataset de calidad del agua
Terraclimate_training_df = pd.concat(list(result_dfs.values()), axis=1)

# Agregar coordenadas y fecha para poder verificar alineación al combinar datasets
Terraclimate_training_df['Latitude']    = Water_Quality_df['Latitude'].values
Terraclimate_training_df['Longitude']   = Water_Quality_df['Longitude'].values
Terraclimate_training_df['Sample Date'] = Water_Quality_df['Sample Date'].values

Terraclimate_training_df = Terraclimate_training_df[[
    'Latitude', 'Longitude', 'Sample Date',
    'pet', 'ppt', 'soil', 'aet', 'water_deficit'
]]

Terraclimate_training_df.to_csv('terraclimate_features_training_v3.csv', index=False)
display(Terraclimate_training_df.head())

In [ ]:
# Subir el archivo al workspace de Snowflake para usarlo en el notebook de modelado
Terraclimate_training_df.to_csv("/tmp/terraclimate_features_training_v3.csv", index=False)
session.sql("""
    PUT file:///tmp/terraclimate_features_training_v3.csv
    'snow://workspace/USER$.PUBLIC."ChallengeEY_prueba2"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print("TerraClimate entrenamiento guardado")

In [ ]:
# Cargar el template de validación (puntos para los que hay que predecir)
Validation_df = pd.read_csv('submission_template.csv')
display(Validation_df.head())
print(Validation_df.shape)

In [ ]:
# Extraer las mismas variables climáticas para los puntos de validación
# Se reutiliza la grilla filtrada del entrenamiento si está disponible en memoria
result_val_dfs = {}

for var in VARIABLES:
    path = f"tc_val_{var}.csv"
    if os.path.exists(path):
        print(f"Cargando {var.upper()} validación desde archivo")
        result_val_dfs[var] = pd.read_csv(path)
    else:
        print(f"Extrayendo {var.upper()} para validación...")
        tc_grid = filterg(ds, var)
        var_df  = assign_nearest_climate(Validation_df, tc_grid, var)
        var_df.to_csv(path, index=False)
        result_val_dfs[var] = var_df

result_val_dfs['water_deficit'] = pd.DataFrame({
    'water_deficit': result_val_dfs['pet']['pet'].values - result_val_dfs['aet']['aet'].values
})

for var, df in result_val_dfs.items():
    assert len(df) == len(Validation_df),         f"Desfase en {var} validación: {len(df)} vs {len(Validation_df)}"
    print(f"{var}: {len(df)} filas OK")

In [ ]:
# Construir tabla de validación con el mismo formato que el de entrenamiento
Terraclimate_validation_df = pd.concat(list(result_val_dfs.values()), axis=1)

Terraclimate_validation_df['Latitude']    = Validation_df['Latitude'].values
Terraclimate_validation_df['Longitude']   = Validation_df['Longitude'].values
Terraclimate_validation_df['Sample Date'] = Validation_df['Sample Date'].values

Terraclimate_validation_df = Terraclimate_validation_df[[
    'Latitude', 'Longitude', 'Sample Date',
    'pet', 'ppt', 'soil', 'aet', 'water_deficit'
]]

Terraclimate_validation_df.to_csv('terraclimate_features_validation_v3.csv', index=False)
display(Terraclimate_validation_df.head())

In [ ]:
Terraclimate_validation_df.to_csv("/tmp/terraclimate_features_validation_v3.csv", index=False)
session.sql("""
    PUT file:///tmp/terraclimate_features_validation_v3.csv
    'snow://workspace/USER$.PUBLIC."ChallengeEY_prueba2"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print("TerraClimate validación guardada")